#STEP 1 — MOUNT DRIVE

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


#STEP 2 — IMPORTS

In [2]:
import os
import cv2
import random
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from torchvision import transforms

import timm

#STEP 3 — DEFINE PATHS

In [3]:
BASE_DIR = "/content/drive/MyDrive/Deepfake_Project"

REAL_DIR = os.path.join(
    BASE_DIR,
    "retraining_dataset/real"
)

FAKE_DIR = os.path.join(
    BASE_DIR,
    "retraining_dataset/fake"
)

MODEL_SAVE_PATH = os.path.join(
    BASE_DIR,
    "trained_models",
    "improved_video_model.pth"
)

#STEP 4 — DEVICE

In [4]:
device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"
)

print(device)

cpu


#STEP 5 — STRONG AUGMENTATIONS

In [5]:
train_transform = transforms.Compose([

    transforms.ToPILImage(),

    transforms.Resize((224,224)),

    transforms.ColorJitter(

        brightness=0.3,

        contrast=0.3,

        saturation=0.3
    ),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(5),

    transforms.GaussianBlur(3),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[0.485,0.456,0.406],

        std=[0.229,0.224,0.225]
    )
])

#STEP 6 — CREATE DATASET CLASS

In [6]:
class VideoDataset(Dataset):

    def __init__(
        self,
        real_dir,
        fake_dir,
        transform=None
    ):

        self.samples = []

        self.transform = transform

        # =========================
        # REAL
        # =========================

        for video in os.listdir(real_dir):

            if video.endswith(".mp4"):

                self.samples.append(

                    (
                        os.path.join(real_dir, video),
                        0
                    )
                )

        # =========================
        # FAKE
        # =========================

        for video in os.listdir(fake_dir):

            if video.endswith(".mp4"):

                self.samples.append(

                    (
                        os.path.join(fake_dir, video),
                        1
                    )
                )

    def __len__(self):

        return len(self.samples)

    def extract_frame(
        self,
        video_path
    ):

        cap = cv2.VideoCapture(video_path)

        total_frames = int(

            cap.get(
                cv2.CAP_PROP_FRAME_COUNT
            )
        )

        if total_frames <= 0:

            cap.release()

            return np.zeros(
                (224,224,3),
                dtype=np.uint8
            )

        frame_idx = random.randint(
            0,
            total_frames - 1
        )

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            frame_idx
        )

        ret, frame = cap.read()

        cap.release()

        if not ret:

            return np.zeros(
                (224,224,3),
                dtype=np.uint8
            )

        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        return frame

    def __getitem__(
        self,
        idx
    ):

        video_path, label = self.samples[idx]

        frame = self.extract_frame(
            video_path
        )

        if self.transform:

            frame = self.transform(frame)

        return frame, label

#STEP 7 — CREATE DATASET

In [7]:
dataset = VideoDataset(

    REAL_DIR,
    FAKE_DIR,

    transform=train_transform
)

print(
    "Dataset Size:",
    len(dataset)
)

Dataset Size: 901


#STEP 8 — CREATE DATALOADER

In [8]:
loader = DataLoader(

    dataset,

    batch_size=8,

    shuffle=True,

    num_workers=0
)

#STEP 9 — LOAD OLD RESNEXT MODEL

In [9]:
video_model = timm.create_model(

    "resnext101_32x8d",

    pretrained=False,

    num_classes=2
)

checkpoint = torch.load(

    "/content/drive/MyDrive/Multimodal_Deepfake_Project/video_assets/best_resnext101_deepfake.pth",

    map_location=device
)

video_model.load_state_dict(

    checkpoint,

    strict=False
)

video_model = video_model.to(device)

print("Old model loaded successfully")

Old model loaded successfully


#STEP 1 — CREATE FRAME FOLDERS

In [10]:
import os

FRAME_REAL_DIR = "/content/drive/MyDrive/Deepfake_Project/frame_dataset/real"

FRAME_FAKE_DIR = "/content/drive/MyDrive/Deepfake_Project/frame_dataset/fake"

os.makedirs(
    FRAME_REAL_DIR,
    exist_ok=True
)

os.makedirs(
    FRAME_FAKE_DIR,
    exist_ok=True
)

print("Frame folders ready")

Frame folders ready


#STEP 2 — FRAME EXTRACTION FUNCTION

In [11]:
import cv2
import random

def extract_random_frames(

    video_path,

    save_dir,

    num_frames=8
):

    cap = cv2.VideoCapture(video_path)

    total_frames = int(

        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    if total_frames <= 0:

        cap.release()

        return

    frame_indices = random.sample(

        range(total_frames),

        min(num_frames, total_frames)
    )

    video_name = os.path.splitext(

        os.path.basename(video_path)

    )[0]

    for idx, frame_id in enumerate(frame_indices):

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            frame_id
        )

        ret, frame = cap.read()

        if ret:

            save_path = os.path.join(

                save_dir,

                f"{video_name}_{idx}.jpg"
            )

            cv2.imwrite(
                save_path,
                frame
            )

    cap.release()

#STEP 3 — DEFINE VIDEO DATASET PATHS

In [12]:
REAL_VIDEO_DIR = "/content/drive/MyDrive/Deepfake_Project/retraining_dataset/real"

FAKE_VIDEO_DIR = "/content/drive/MyDrive/Deepfake_Project/retraining_dataset/fake"

#STEP 4 — EXTRACT REAL FRAMES

In [13]:
for video in os.listdir(REAL_VIDEO_DIR):

    if video.endswith(".mp4"):

        extract_random_frames(

            os.path.join(
                REAL_VIDEO_DIR,
                video
            ),

            FRAME_REAL_DIR,

            num_frames=3
        )

print("Real frames extracted")

Real frames extracted


#STEP 5 — EXTRACT FAKE FRAMES

In [14]:
for video in os.listdir(FAKE_VIDEO_DIR):

    if video.endswith(".mp4"):

        extract_random_frames(

            os.path.join(
                FAKE_VIDEO_DIR,
                video
            ),

            FRAME_FAKE_DIR,

            num_frames=3
        )

print("Fake frames extracted")

Fake frames extracted


#STEP 6 — NEW FRAME DATASET CLASS

In [15]:
from PIL import Image

class FrameDataset(Dataset):

    def __init__(
        self,
        real_dir,
        fake_dir,
        transform=None
    ):

        self.samples = []

        self.transform = transform

        # =========================
        # REAL
        # =========================

        for img in os.listdir(real_dir):

            if img.endswith(".jpg"):

                self.samples.append(

                    (
                        os.path.join(real_dir, img),
                        0
                    )
                )

        # =========================
        # FAKE
        # =========================

        for img in os.listdir(fake_dir):

            if img.endswith(".jpg"):

                self.samples.append(

                    (
                        os.path.join(fake_dir, img),
                        1
                    )
                )

    def __len__(self):

        return len(self.samples)

    def __getitem__(
        self,
        idx
    ):

        img_path, label = self.samples[idx]

        image = Image.open(
            img_path
        ).convert("RGB")

        image = np.array(image)

        if self.transform:

            image = self.transform(image)

        return image, label

#STEP 7 — DEFINE FRAME PATHS

In [16]:
FRAME_REAL_DIR = "/content/drive/MyDrive/Deepfake_Project/frame_dataset/real"

FRAME_FAKE_DIR = "/content/drive/MyDrive/Deepfake_Project/frame_dataset/fake"

#STEP 8 — CREATE DATASET

In [17]:
dataset = FrameDataset(

    FRAME_REAL_DIR,
    FRAME_FAKE_DIR,

    transform=train_transform
)

print(
    "Dataset Size:",
    len(dataset)
)

Dataset Size: 2703


#STEP 9 — CREATE LIGHTWEIGHT DATALOADER

In [18]:
loader = DataLoader(

    dataset,

    batch_size=16,

    shuffle=True,

    num_workers=0
)

#STEP 10 — LOSS + OPTIMIZER

In [19]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(

    video_model.parameters(),

    lr=1e-5
)

#STEP 11 — TRAINING LOOP

In [20]:
EPOCHS = 3

for epoch in range(EPOCHS):

    video_model.train()

    total_loss = 0

    correct = 0

    total = 0

    loop = tqdm(loader)

    for frames, labels in loop:

        frames = frames.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = video_model(frames)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        preds = outputs.argmax(dim=1)

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

        acc = correct / total

        loop.set_description(
            f"Epoch {epoch+1}"
        )

        loop.set_postfix(

            loss=loss.item(),

            acc=acc
        )

    print(

        f"\nEpoch {epoch+1} Accuracy: {acc:.4f}"
    )

Epoch 1: 100%|██████████| 169/169 [1:40:55<00:00, 35.83s/it, acc=0.558, loss=0.68]



Epoch 1 Accuracy: 0.5575


Epoch 2: 100%|██████████| 169/169 [1:40:44<00:00, 35.77s/it, acc=0.671, loss=0.632]



Epoch 2 Accuracy: 0.6711


Epoch 3: 100%|██████████| 169/169 [1:39:55<00:00, 35.47s/it, acc=0.753, loss=0.512]


Epoch 3 Accuracy: 0.7529


#STEP 12 — SAVE IMPROVED MODEL

In [21]:
torch.save(

    video_model.state_dict(),

    MODEL_SAVE_PATH
)

print("Improved model saved")

Improved model saved
